# 04 — Fine-tuning (SEC Filings)

Continues training the pretrained checkpoint from `03_pretrain.ipynb`
(step 14,999) on **only** the 158 real SEC 10-K/10-Q filings collected in
`01_data_collection.ipynb` — same architecture (imported from
[`model.py`](model.py), not redefined), lower learning rate (10x lower
than pretraining, so it nudges the model toward SEC-filing style rather
than overwriting what it already learned about English).

**Fine-tuning:** 7.4M SEC-filing tokens, 3,000 steps, lr 3e-5 —
**loss dropped from 5.78 → 2.77** (train) over the run. Fine-tuned
checkpoints are saved to a separate directory from the pretrained ones, so
the pretrained checkpoint is never overwritten.

Requires `model_step14999.pt` from `03_pretrain.ipynb` — not committed to
this repo (see that notebook for why), so re-run pretraining first if
starting fresh.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q tokenizers torch
print("Ready.")

Mounted at /content/drive
Ready.

In [1]:
import os
import torch
import numpy as np
from tokenizers import Tokenizer

# --- Load tokenizer ---
tokenizer_path = '/content/drive/MyDrive/sec_chatbot/tokenizer/tokenizer.json'
tokenizer = Tokenizer.from_file(tokenizer_path)
vocab_size = tokenizer.get_vocab_size()
print(f"Tokenizer loaded. Vocabulary size: {vocab_size}")

# --- Collect all SEC filings ---
data_dir = '/content/drive/MyDrive/sec_chatbot/data'
filing_paths = []
for company in os.listdir(data_dir):
    company_path = os.path.join(data_dir, company)
    if os.path.isdir(company_path):
        for f in os.listdir(company_path):
            if f.endswith('.txt'):
                filing_paths.append(os.path.join(company_path, f))
print(f"Filings found: {len(filing_paths)}")

# --- Encode file by file to keep memory usage low ---
all_ids = []
print("Encoding filings...")
for i, path in enumerate(filing_paths):
    with open(path, 'r', encoding='utf-8') as f:
        text = f.read()
    ids = tokenizer.encode(text).ids
    all_ids.append(np.array(ids, dtype=np.uint16))
    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(filing_paths)} filings encoded")

data_np = np.concatenate(all_ids)
del all_ids
data = torch.from_numpy(data_np.astype(np.int64))
del data_np
print(f"\nTotal tokens: {len(data):,}")

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"Training tokens: {len(train_data):,}")
print(f"Validation tokens: {len(val_data):,}")

Tokenizer loaded. Vocabulary size: 8000
Filings found: 158
Encoding filings...
  25/158 filings encoded
  50/158 filings encoded
  75/158 filings encoded
  100/158 filings encoded
  125/158 filings encoded
  150/158 filings encoded

Total tokens: 7,408,607
Training tokens: 6,667,746
Validation tokens: 740,861

In [1]:
import sys
sys.path.append('/content/drive/MyDrive/sec_chatbot/scripts')  # wherever model.py lives
from model import GPTLanguageModel, block_size

batch_size = 16
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

train_data = train_data.to(device)
val_data = val_data.to(device)

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

model = GPTLanguageModel(vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model created with {n_params/1e6:.1f} million parameters")

Using device: cuda
Model created with 33.5 million parameters

In [1]:
import glob

# --- Load the pretrained weights ---
ckpt_dir = '/content/drive/MyDrive/sec_chatbot/checkpoints'
checkpoints = glob.glob(os.path.join(ckpt_dir, 'model_step*.pt'))
latest = max(checkpoints, key=lambda p: int(p.split('step')[1].split('.')[0]))
print(f"Loading pretrained model: {latest}")

ckpt = torch.load(latest)
model.load_state_dict(ckpt['model_state'])
print(f"Loaded weights from step {ckpt['step']} (val loss was {ckpt['val_loss']:.4f})")

# --- Fine-tuning settings ---
max_iters = 3000
eval_interval = 250
eval_iters = 100
learning_rate = 3e-5   # 10x lower than pretraining -- nudge, don't overwrite

# Fresh optimizer -- we do NOT reuse the pretraining optimizer state;
# this is a new task with its own gradient momentum.
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Separate checkpoint directory so fine-tuning never overwrites the
# pretrained model.
ft_dir = '/content/drive/MyDrive/sec_chatbot/checkpoints_finetuned'
os.makedirs(ft_dir, exist_ok=True)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# --- Fine-tuning loop ---
print("\nStarting fine-tuning on SEC filings...\n")
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"Step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        ckpt_path = os.path.join(ft_dir, f'finetuned_step{iter}.pt')
        torch.save({
            'step': iter,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'train_loss': losses['train'].item(),
            'val_loss': losses['val'].item(),
        }, ckpt_path)

    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("\nFine-tuning complete.")

Loading pretrained model: /content/drive/MyDrive/sec_chatbot/checkpoints/model_step14999.pt
Loaded weights from step 14999 (val loss was 3.8494)

Starting fine-tuning on SEC filings...

Step 0: train loss 5.7818, val loss 5.6581
Step 250: train loss 4.1557, val loss 4.3167
Step 500: train loss 3.7943, val loss 4.0468
Step 750: train loss 3.5944, val loss 3.8543
Step 1000: train loss 3.4134, val loss 3.7523
Step 1250: train loss 3.2867, val loss 3.6621
Step 1500: train loss 3.1893, val loss 3.5963
Step 1750: train loss 3.0726, val loss 3.5319
Step 2000: train loss 3.0047, val loss 3.5159
Step 2250: train loss 2.9493, val loss 3.4341
Step 2500: train loss 2.8581, val loss 3.4040
Step 2750: train loss 2.8171, val loss 3.3682
Step 2999: train loss 2.7697, val loss 3.3670

Fine-tuning complete.

**Loss dropped sharply and immediately** (5.78 → 4.16 in the first 250
steps alone) — the model rapidly adapted to SEC-filing vocabulary and
phrasing, which is exactly what fine-tuning a general-English pretrained
model on a narrow domain should look like.

In [1]:
from torch.nn import functional as F

@torch.no_grad()
def generate(model, max_new_tokens=200):
    bos_id = tokenizer.token_to_id("<BOS>")
    idx = torch.tensor([[bos_id]], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
    return tokenizer.decode(idx[0].tolist())

print("--- GENERATED TEXT (fine-tuned on SEC filings) ---\n")
print(generate(model, max_new_tokens=250))

--- GENERATED TEXT (fine-tuned on SEC filings) ---

of intangible assets , range second - of - U near ned assets , liabilities , time obligation or present ations for quant itative and reporting activities . General and administrative expend iture expenses include the Registrant 's fiscal years Jul August 1 , 2023 ( in millions ) %, Amount ( 20 ) Other senior notes Total automotive sales : Comp ensation 5 , 4 50 amortization of intangible assets : Product s $ 6 , 9 34 $ 1 . 8 %, 18 90 35 2 , 500 19 2 Total other current assets $ 1 , 5 29 $ 2 . 5 $ 1 , 5 29 Int angible assets and liabilities 38 1 During the Equity customers ' total exchange rate : Product assets 24 2 28 . Deferred revenue : The following is a period from beginning of period to each quarter of two type were : lar Months Ended September 30 , 2023 Six Months Ended September 30 , 2021 2022 2022 Ret ained and bonds $ 12 , 9 23 $ 16 , 2 35 Interest obligations 1 , 24 1 1 , 78 5 36 , 7 69 6 50 40 9 7 , 5 60 57 1 , 88 7 New Yor

Notice the vocabulary shift from the pretrained sample in
`03_pretrain.ipynb` — this output is dense with SEC-filing-specific terms
("intangible assets", "amortization", "Registrant", "Deferred revenue",
tabular dollar figures) that simply didn't appear in the Wikipedia-style
pretraining sample. Coherence is still limited at this parameter count,
but the domain adaptation is unmistakable.

In [1]:
# --- Corpus quality check: how much of the fine-tuning data is prose
# vs. dense financial tables (which read very differently to a token
# predictor)? ---
import random

sample_path = random.choice(filing_paths)
print(f"Sampling from: {os.path.basename(sample_path)}\n")

with open(sample_path, 'r', encoding='utf-8') as f:
    text = f.read()

chunks = text.split('. ')
print(f"Total chunks: {len(chunks)}\n")
print("--- 15 RANDOM CHUNKS ---\n")
for chunk in random.sample(chunks, 15):
    if len(chunk) == 0:
        continue
    digit_frac = sum(c.isdigit() for c in chunk) / len(chunk)
    print(f"[digits: {digit_frac:.0%}] {chunk[:150]}")
    print()

Sampling from: 2024-02-02_10Q_aapl-20231230.txt

--- 15 RANDOM CHUNKS ---

al statements are unaudited and, in our opinion, include all adjustments, consisting of normal recurring adjustments and acc

[digits: 0%] We seek to expand our fulfillment network to accommodate a greater selection and in-stock inventory levels and to meet anticipated shipment volumes fr

[digits: 3%] [X] Indicate by check mark whether the registrant is a shell company (as defined in Rule 12b-2 of the Exchange Act)

[digits: 0%] Failure to realize the benefits of amounts we invest in new technologies, products, or services could result in the value of those investments being w

[digits: 0%] These regulations and laws cover taxation, privacy, data use, data protection, data security, data localization, network security, consumer protection

[digits: 13%] There were no borrowings outstanding under the 2023 Short-Term Credit Agreement as of December 31, 2023 and September 30, 2024

In [1]:
# --- Same measurement across the whole corpus, not just one filing ---
total_chars = 0
table_chars = 0
prose_chars = 0

for path in filing_paths:
    with open(path, 'r', encoding='utf-8') as f:
        text = f.read()
    chunks = text.split('. ')
    for chunk in chunks:
        if len(chunk) == 0:
            continue
        digit_frac = sum(c.isdigit() for c in chunk) / len(chunk)
        total_chars += len(chunk)
        if digit_frac > 0.15:      # more than 15% digits ~= likely a table fragment
            table_chars += len(chunk)
        else:
            prose_chars += len(chunk)

print(f"Total characters:  {total_chars:,}")
print(f"Table-like:        {table_chars:,}  ({table_chars/total_chars:.1%})")
print(f"Prose:             {prose_chars:,}  ({prose_chars/total_chars:.1%})")

Total characters:  34,342,532
Table-like:        3,022,898  (8.8%)
Prose:             31,319,634  (91.2%)

In [1]:
@torch.no_grad()
def generate_from_prompt(model, prompt, max_new_tokens=150):
    ids = tokenizer.encode(prompt).ids
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
    return tokenizer.decode(idx[0].tolist())

prompts = [
    "Our business faces significant risks including",
    "Revenue increased during the quarter primarily due to",
    "The Company is subject to legal proceedings arising",
]

for p in prompts:
    print(f"--- PROMPT: {p}\n")
    print(generate_from_prompt(model, p, max_new_tokens=120))
    print("\n" + "="*60 + "\n")

--- PROMPT: Revenue increased during the quarter primarily due to

Revenue increased during the quarter primarily due to higher dis crete err ors . These activities ex cluded compensation option investments were susp ended either due to higher investment strategy prior to the first quarter of fiscal joining , partially increase to investment rates . The increases were driven by factors in the prior period in con form ity center prices rated technologies due to lower costs associated with lower earnings . The result in change in fair value gross margin for the portion of which increased $ 4 19 million , or $ 3 12 million , and would increase in the three fiscal quarters of fiscal 2024 . Revenues received mostly by credit losses , which primarily impacted , with higher profit ability . The non


--- PROMPT: The Company is subject to legal proceedings arising

The Company is subject to legal proceedings ar ising from consul t ants , claims and measurement we , which could have the experie

**What this demonstrates:** given the start of a real SEC-filing sentence,
the model continues with financially-plausible vocabulary, phrase
patterns, and sentence structure it learned purely from the fine-tuning
corpus — dollar figures, fiscal-period references, legal/compliance
phrasing — without ever being given the actual continuation. It's not
fluent or factually grounded (a 33.5M-parameter model trained on ~7.4M
domain tokens isn't going to be), but it's clear evidence the model
specialized to this domain rather than just memorizing Wikipedia.

This model was later used as a technical foundation in a larger
application — see the root [README](README.md) for the honest distinction
between what this model demonstrates and how that application actually
answers questions in production.